<a href="https://colab.research.google.com/github/0110225150-hub/Nunaaaaa/blob/main/Proyek_Tim_Pertemuan4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**10.1 — Muat dataset proyek tim**

In [ ]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'student_career_success_dataset.csv'
if not os.path.exists(FILE):
    try:                # <--- 4 spasi
        from google.colab import files  # <--- 8 spasi
        files.upload()                  # <--- 8 spasi
    except ImportError: # <--- 4 spasi (sejajar dengan try)
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print('Berhasil dimuat:', df.shape)

Saving student_career_success_dataset.csv to student_career_success_dataset.csv
Berhasil dimuat: (50000, 29)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             50000 non-null  object 
 1   Age                    50000 non-null  int64  
 2   Gender                 50000 non-null  object 
 3   University_Year        50000 non-null  object 
 4   Major                  50000 non-null  object 
 5   Attendance_Percentage  50000 non-null  int64  
 6   Study_Hours_Per_Week   50000 non-null  int64  
 7   CGPA                   50000 non-null  float64
 8   Academic_Performance   50000 non-null  object 
 9   Programming_Skill      50000 non-null  int64  
 10  Projects_Completed     50000 non-null  int64  
 11  Certifications         50000 non-null  int64  
 12  Hackathons             50000 non-null  int64  
 13  GitHub_Profile         50000 non-null  object 
 14  Internships            50000 non-null  int64  
 15  Le

**10.2 - Jalankan kesembilan pemeriksaan pada tim**

In [ ]:
# 10.2 — jalankan kesembilan pemeriksaan pada data tim

# 1. Potret awal: shape, info, describe
print("=== 1. POTRET AWAL ===")
print("Ukuran dataset (shape):", df.shape)
print("\nDescribe Kolom Numerik Utama:")
display(df[['CGPA', 'Attendance_Percentage', 'Employability_Score', 'Starting_Salary_USD']].describe().round(2))
print("\nDescribe Kolom Teks/Kategori:")
display(df.describe(include='object').T)


=== 1. POTRET AWAL ===
Ukuran dataset (shape): (50000, 29)

Describe Kolom Numerik Utama:


,CGPA,Attendance_Percentage,Employability_Score,Starting_Salary_USD
count,50000.00,50000.00,50000.00,50000.00
mean,3.11,81.41,213.99,68988.17
std,0.29,9.71,26.82,38480.61
min,2.00,50.00,82.30,0.00
25%,2.91,75.00,197.35,65229.25
50%,3.11,82.00,217.25,83044.50
75%,3.31,88.00,233.60,96486.25
max,4.00,100.00,279.05,110000.00



Describe Kolom Teks/Kategori:


,count,unique,top,freq
Student_ID,50000,50000,ST049984,1
Gender,50000,3,Male,24533
University_Year,50000,4,Junior,14870
Major,50000,8,Computer Science,12407
Academic_Performance,50000,4,Average,28890
GitHub_Profile,50000,2,Yes,40680
Leadership_Experience,50000,2,No,30403
LinkedIn_Profile,50000,2,Yes,35917
English_Proficiency,50000,3,Advanced,39640
Placement_Status,50000,2,Placed,39040


In [ ]:

# 2. Nilai kosong sejati
print("\n=== 2. NILAI KOSONG SEJATI ===")
kosong_sejati = df.isna().sum()
print("Jumlah sel kosong sejati (isna) per kolom:")
print(kosong_sejati[kosong_sejati > 0] if kosong_sejati.sum() > 0 else "Tidak ada sel kosong sejati (semua terisi/menggunakan placeholder).")



=== 2. NILAI KOSONG SEJATI ===
Jumlah sel kosong sejati (isna) per kolom:
Tidak ada sel kosong sejati (semua terisi/menggunakan placeholder).


In [ ]:

# 3. Placeholder teks dan angka
print("\n=== 3. PLACEHOLDER TEKS DAN ANGKA ===")
placeholders = {
    'Company_Tier (No Company)': (df['Company_Tier'] == 'No Company').sum(),
    'Career_Field (Not Placed)': (df['Career_Field'] == 'Not Placed').sum(),
    'Placement_Mode (Not Applicable)': (df['Placement_Mode'] == 'Not Applicable').sum(),
    'Starting_Salary_USD (0)': (df['Starting_Salary_USD'] == 0).sum()
}
for k, v in placeholders.items():
    print(f"- {k:35s}: {v:6d} baris ({v/len(df)*100:.2f}%)")



=== 3. PLACEHOLDER TEKS DAN ANGKA ===
- Company_Tier (No Company)          :  10960 baris (21.92%)
- Career_Field (Not Placed)          :  10960 baris (21.92%)
- Placement_Mode (Not Applicable)    :  10960 baris (21.92%)
- Starting_Salary_USD (0)            :  10960 baris (21.92%)


In [ ]:

# 4. Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong
print("\n=== 4. POLA KEKOSONGAN (MAR / MCAR / MNAR) ===")
print("Pemeriksaan kekosongan Company_Tier berdasarkan Placement_Status (%):")
display(pd.crosstab(df['Placement_Status'], df['Company_Tier'] == 'No Company', normalize='index') * 100)
print("Kesimpulan: Kekosongan bersifat MAR (Missing At Random) karena 100% bergantung pada Placement_Status == 'Not Placed'.")



=== 4. POLA KEKOSONGAN (MAR / MCAR / MNAR) ===
Pemeriksaan kekosongan Company_Tier berdasarkan Placement_Status (%):


Company_Tier,False,True
Placement_Status,,
Not Placed,0.0,100.0
Placed,100.0,0.0


Kesimpulan: Kekosongan bersifat MAR (Missing At Random) karena 100% bergantung pada Placement_Status == 'Not Placed'.


In [ ]:

# 5. Duplikat persis dan duplikat menurut kunci
print("\n=== 5. DUPLIKAT ===")
print("Baris duplikat persis seluruh kolom :", df.duplicated().sum())
print("Duplikat menurut Student_ID  :", df['Student_ID'].duplicated(keep=False).sum())


=== 5. DUPLIKAT ===
Baris duplikat persis seluruh kolom : 0
Duplikat menurut Student_ID  : 0


In [ ]:


# 6. Konsistensi kolom teks
print("\n=== 6. KONSISTENSI KOLOM TEKS ===")
inkonsisten = 0
for kol in df.select_dtypes(include='object').columns:
    s = df[kol].astype(str)
    mentah = s.nunique()
    rapi = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        print(f"Peringatan: {kol} punya perbedaan spasi/kapitalisasi ({mentah} -> {rapi})")
        inkonsisten += 1
if inkonsisten == 0:
    print("Semua kolom teks bersih (tidak ada spasi di awal/akhir atau perbedaan kapitalisasi abnormal).")



=== 6. KONSISTENSI KOLOM TEKS ===
Semua kolom teks bersih (tidak ada spasi di awal/akhir atau perbedaan kapitalisasi abnormal).


In [ ]:

# 7. Tipe data yang salah
print("\n=== 7. TIPE DATA YANG SALAH ===")
print("Pemeriksaan tipe data kolom:")
print(df.dtypes.value_counts())
print("Kesimpulan: Tipe data angka (int64/float64) dan teks (object) sudah sesuai.")



=== 7. TIPE DATA YANG SALAH ===
Pemeriksaan tipe data kolom:
int64      14
object     13
float64     2
Name: count, dtype: int64
Kesimpulan: Tipe data angka (int64/float64) dan teks (object) sudah sesuai.


In [ ]:

# 8. Aturan rentang (Aturan Validitas)
print("\n=== 8. ATURAN RENTANG (VALIDITAS) ===")
aturan_rentang = {
    'CGPA di luar 0.0 - 4.0': ((df['CGPA'] < 0) | (df['CGPA'] > 4.0)).sum(),
    'Attendance % di luar 0 - 100': ((df['Attendance_Percentage'] < 0) | (df['Attendance_Percentage'] > 100)).sum(),
    'Resume Score di luar 0 - 100': ((df['Resume_Score'] < 0) | (df['Resume_Score'] > 100)).sum(),
    'Interview Score di luar 0 - 100': ((df['Interview_Score'] < 0) | (df['Interview_Score'] > 100)).sum(),
    'Skill (1-10) di luar rentang': ((df['Programming_Skill'] < 1) | (df['Programming_Skill'] > 10)).sum()
}
for nama, jumlah in aturan_rentang.items():
    print(f"- {nama:35s}: dilanggar {jumlah} baris")


=== 8. ATURAN RENTANG (VALIDITAS) ===
- CGPA di luar 0.0 - 4.0             : dilanggar 0 baris
- Attendance % di luar 0 - 100       : dilanggar 0 baris
- Resume Score di luar 0 - 100       : dilanggar 0 baris
- Interview Score di luar 0 - 100    : dilanggar 0 baris
- Skill (1-10) di luar rentang       : dilanggar 0 baris


In [ ]:


# 9. Aturan hubungan antarkolom (Akurasi & Logika Bisnis)
print("\n=== 9. ATURAN HUBUNGAN ANTARKOLOM (LOGIKA DOMAIN) ===")
freshman_placed = len(df[(df['University_Year'] == 'Freshman') & (df['Placement_Status'] == 'Placed')])
salary_not_placed = len(df[(df['Placement_Status'] == 'Not Placed') & (df['Starting_Salary_USD'] > 0)])

print(f"- Mhs 'Not Placed' tetapi Gaji > 0 USD       : {salary_not_placed} baris (Konsisten/Valid)")
print(f"- Mhs 'Freshman' (Thn 1) tercatat 'Placed'   : {freshman_placed} baris (Anomali Domain Bisnis!)")


=== 9. ATURAN HUBUNGAN ANTARKOLOM (LOGIKA DOMAIN) ===
- Mhs 'Not Placed' tetapi Gaji > 0 USD       : 0 baris (Konsisten/Valid)
- Mhs 'Freshman' (Thn 1) tercatat 'Placed'   : 7789 baris (Anomali Domain Bisnis!)


**10.3 - Susun Profil Kualitas Data tim**

In [ ]:
# 10.3 — susun profil kualitas data tim
import os

temuan_tim = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan_tim.append({
        'dimensi': dimensi,
        'kolom': kolom,
        'masalah': masalah,
        'jumlah': int(jumlah),
        'persen': round(jumlah / len(df) * 100, 2),
        'keparahan': keparahan,          # Tinggi / Sedang / Rendah
        'rencana': rencana             # Rencana perbaikan untuk Pertemuan 5
    })

# Perekaman temuan hasil diagnosis
catat('Kelengkapan', 'Starting_Salary_USD', 'Nilai 0 USD pada gaji untuk status Not Placed (MAR)', 10960, 'Tinggi', 'Ubah 0 menjadi NaN sebelum menghitung rata-rata gaji nasional')
catat('Validitas', 'University_Year & Placement_Status', 'Anomali domain: Mhs Freshman (Thn 1) tercatat Placed & magang 4x', 7789, 'Tinggi', 'Validasi dengan pengelola data / pisahkan segmen Freshmen dari analisis placement akhir')
catat('Kelengkapan', 'Company_Tier', 'Placeholder "No Company" untuk mahasiswa Not Placed (MAR)', 10960, 'Rendah', 'Tandai "Not Applicable", jangan diisi nama perusahaan')
catat('Kelengkapan', 'Career_Field', 'Placeholder "Not Placed" untuk mahasiswa belum bekerja (MAR)', 10960, 'Rendah', 'Tandai "Not Applicable", abaikan dalam analisis spesialisasi')
catat('Kelengkapan', 'Placement_Mode', 'Placeholder "Not Applicable" untuk mahasiswa Not Placed (MAR)', 10960, 'Rendah', 'Dibiarkan sebagai kategori terpisah (Not Applicable)')
catat('Keunikan', 'Student_ID', 'Duplikasi data kunci utama', 0, 'Rendah', 'Data unik sempurna (50.000 ID unik)')
catat('Konsistensi', 'Kolom Teks', 'Format kapitalisasi atau spasi abnormal', 0, 'Rendah', 'Data sudah konsisten')

# Menyusun DataFrame profil
profil_tim = pd.DataFrame(temuan_tim).sort_values(['keparahan', 'jumlah'], ascending=[True, False])

# Membuat folder reports jika belum ada
os.makedirs('reports', exist_ok=True)

# Simpan profil kualitas data ke CSV
profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)

print("=== PROFIL KUALITAS DATA TIM ===")
display(profil_tim)
print("\nProfil kualitas data berhasil tersimpan di 'reports/profil_kualitas_data.csv'!")


=== PROFIL KUALITAS DATA TIM ===


,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
2,Kelengkapan,Company_Tier,"Placeholder ""No Company"" untuk mahasiswa Not P...",10960,21.92,Rendah,"Tandai ""Not Applicable"", jangan diisi nama per..."
3,Kelengkapan,Career_Field,"Placeholder ""Not Placed"" untuk mahasiswa belum...",10960,21.92,Rendah,"Tandai ""Not Applicable"", abaikan dalam analisi..."
4,Kelengkapan,Placement_Mode,"Placeholder ""Not Applicable"" untuk mahasiswa N...",10960,21.92,Rendah,Dibiarkan sebagai kategori terpisah (Not Appli...
5,Keunikan,Student_ID,Duplikasi data kunci utama,0,0.00,Rendah,Data unik sempurna (50.000 ID unik)
6,Konsistensi,Kolom Teks,Format kapitalisasi atau spasi abnormal,0,0.00,Rendah,Data sudah konsisten
0,Kelengkapan,Starting_Salary_USD,Nilai 0 USD pada gaji untuk status Not Placed ...,10960,21.92,Tinggi,Ubah 0 menjadi NaN sebelum menghitung rata-rat...
1,Validitas,University_Year & Placement_Status,Anomali domain: Mhs Freshman (Thn 1) tercatat ...,7789,15.58,Tinggi,Validasi dengan pengelola data / pisahkan segm...



Profil kualitas data berhasil tersimpan di 'reports/profil_kualitas_data.csv'!


In [ ]:
import numpy as np
bersih = df.copy()

# Keputusan 1: 0 USD = placeholder -> NaN
bersih['Starting_Salary_USD'] = bersih['Starting_Salary_USD'].replace(0, np.nan)
# Keputusan 2: seragamkan placeholder kategori
bersih['Company_Tier'] = bersih['Company_Tier'].replace('No Company', 'Not Applicable')
bersih['Career_Field'] = bersih['Career_Field'].replace('Not Placed', 'Not Applicable')
# Keputusan 3: penanda anomali, baris tdk dihapus
bersih['Flag_Freshman_Placed'] = ((bersih['University_Year'] == 'Freshman') &
                                  (bersih['Placement_Status'] == 'Placed')).astype(int)

print('Baris/kolom      :', df.shape, '->', bersih.shape)
print('Total sel kosong :', int(df.isna().sum().sum()), '->', int(bersih.isna().sum().sum()))
print('Duplikat         :', int(df.duplicated().sum()), '->', int(bersih.duplicated().sum()))
for k in ['Starting_Salary_USD', 'Employability_Score', 'Resume_Score']:
    print(k, 'mean/median/std sebelum:', df[k].agg(['mean','median','std']).round(2).tolist(),
          'sesudah:', bersih[k].agg(['mean','median','std']).round(2).tolist())
os.makedirs('data', exist_ok=True)
bersih.to_csv('data/student_career_success_bersih.csv', index=False)

Baris/kolom      : (50000, 29) -> (50000, 30)
Total sel kosong : 0 -> 10960
Duplikat         : 0 -> 0
Starting_Salary_USD mean/median/std sebelum: [68988.17, 83044.5, 38480.61] sesudah: [88355.75, 89025.0, 13608.5]
Employability_Score mean/median/std sebelum: [213.99, 217.25, 26.82] sesudah: [213.99, 217.25, 26.82]
Resume_Score mean/median/std sebelum: [96.05, 100.0, 7.33] sesudah: [96.05, 100.0, 7.33]
